# R_M45 -- ML synthetic control

Model M45 (ML synthetic control) in R, on the panel R_P00 prepared, with the design set in THIS notebook (v20.57: applied when it runs). Every result carries the design-based SE (the honest benchmark) and is flagged if its SE is ~0.

In [ ]:
# the bundle's R folder (works from rstudio/, jupyter/ or the project root)
R_HOME_DIR <- normalizePath(if (file.exists("lib/reward_paths.R")) "." else "..", winslash = "/")
for (f in c("reward_paths.R", "reward_design.R", "reward_prep.R", "reward_models_core.R")) source(file.path(R_HOME_DIR, "lib", f), local = environment())
cat("data root:", ROOT, "| output:", OUTPUT_DIR, "\n")
invisible(confirm_packages())   # v20.55: are ALL R packages installed? (pre-built first; a missing one is installed now when AUTO_INSTALL_PACKAGES)

**Settings** -- the outcome (`ALL_OUTCOMES <- TRUE` runs every outcome) and THE DESIGN OF THIS RUN (v20.57): timing (your fund workbook by default), dose, control rings, years, seasons, fragments, overlaps, fixed effects, covariates -- applied when this model runs, on the panel R_P00 built once. Change them here and re-run this notebook only.

In [ ]:
OUTCOME      <- "NDVI"
ALL_OUTCOMES <- FALSE
# ---- v20.57: THE DESIGN OF THIS RUN -- every option below is applied HERE, when this model runs, on the panel R_P00 built ONCE.
#      Change any of them and re-run this notebook: R_P00 is never re-run for it. What you set is what runs -- DESIGN IN EFFECT
#      (printed by model_design(), saved as DESIGN_IN_EFFECT.csv next to the results) lists each option, the value used and why.
DESIGN_MODE      <- "recommended"   # "recommended": an option set to "data" is chosen FROM THE DATA (DESIGN_RECOMMENDATION.md: export breaks,
                                    #   fill years, spillover into inner rings -- never the effect) | "manual": "data" = every ring / every year.
                                    #   A value you set (e.g. CONTROL_RINGS <- 1:3, PRE_YEARS <- 4) is ALWAYS used exactly as set, in both modes
TREATMENT_TIMING <- "fund"          # "fund": each sub-watershed's first treated SEASON from your fund workbook -- the release timing, back-cast
                                    #   before the file's first month, the season AFTER the start (no anticipation): staggered, season-level
                                    #   cohorts (results/FUND/FUND_TIMING_AND_DOSE) | "registry": its year in data/sites/sites.csv | "fixed"
TREATMENT_YEAR   <- 2022            # "fixed" timing -- and the fall-back for a sub-watershed the fund file / registry does not date (announced)
FUND_START_RULE  <- "backcast"      # fund timing: "backcast" | "share" (the amount reaches 10 % of the target) | "file_start" (a lower bound)
DOSE_VARIABLE    <- "dose_intensity_per_ha"   # the dose (fund file): amount released / treatment area | "dose_amount_sws" | "dose_share_of_target"
                                              #   -- 0 on the control rings and before the treatment starts
CONTROL_RINGS    <- "data"          # control ring(s): "data" | 1:5 | 1:3 | 2:5 | c(1, 3, 5)
CLUSTER          <- "auto"          # v20.59: "auto" (the sub-watersheds; fewer than 6 -> the years) | "block" (~1 km spatial blocks of pixels: many clusters, spatial correlation absorbed)
PRE_YEARS        <- "data"          # years BEFORE the start: "data" | NA (every year) | 4 (the 4 years before) | 2015 (a calendar year = the FIRST pre year)
POST_YEARS       <- "data"          # years FROM the start:   "data" | NA (every year) | 2 (the start year + 1) | 2025 (a calendar year = the LAST post year)
SEASONS          <- "all"           # "all" (annual composite + Kharif/Rabi/Zaid: year AND season variation) | "seasonal" | "yearly" |
                                    #   "Rabi" | c("Kharif", "Rabi") | "auto" (the data decide)
EXCLUDE_TRANSITION_YEAR <- FALSE    # TRUE = the first treated year of each series leaves the sample (robustness)
UNIT_FE          <- "pixel_season"  # one fixed effect per pixel x season series (+ year x season effects) | "pixel"
SUB_WATERSHEDS   <- "data"          # v20.58: the sub-watershed(s) THIS run processes: "data" = every one with >= 5 % of the largest one's rows (one
                                    #   export = its own sub-watershed) | "major" (the largest only) | a name or id, e.g. "Koranahalli" | c(1, 7)
FRAGMENT_RULE    <- "drop"          # YOUR RULE: rows of sub-watersheds NOT processed and rows outside every polygon leave EVERY group -- treated and
                                    #   control, pre and post (SAMPLE INTEGRITY confirms it at every run) | "keep"
OVERLAP_ROWS     <- "drop"          # YOUR RULE: overlapping pixels leave every group -- a pixel treated in another processed sub-watershed (as a
                                    #   control), a repeated pixel-year-season, a near-duplicate pixel, a pixel whose ring the exports disagree on | "keep"
POOLED_FE        <- "site_period"   # several sub-watersheds: sub-watershed x year x season effects | "period" (one shared year x season effect)
EXCLUDE_GAPFILLED <- TRUE           # rows the exporter filled from history (GapFilled / Coverage 0) are not observations | FALSE keeps them
OUTCOME_SCREEN    <- "drop"         # v20.59: "drop" (a year-season constant across pixels -- a fill value -- or with collapsed coverage leaves the
                                    #   model; its evidence: OUTCOME_SCREEN_<outcome>.csv beside the results) | "keep" (reported and KEPT; results tagged
                                    #   _screenKept) | "off". Your v20.58 log stopped here ("41 year-seasons are NOT pixel data ... Re-export it")
DESIGN_SOURCE     <- "panel"        # v20.59 -- YOUR RULE: the PANEL's treat / control / pre / post / did (the exports' Treat flag, PERIOD_RULE) are
                                    #   what this model estimates on (results tagged _panelDesign; the design in effect above is COMPARED with them) | "model" =
                                    #   design-based modelling: the timing / TREATMENT_YEAR / transition-year settings above build post / pre / did
COVARIATES       <- c("Rain", "Tmax", "Tmean", "Tmin")   # the weather covariates of this model | character(0) = none
# ================================================================================================================================
# OPTIONAL CUSTOMISATIONS (1 Oct, your rule) -- every one has a USE_ switch. FALSE = NOT applied at all, whatever the value beside it
# says (the standard design above runs as it is); TRUE = applied exactly as set. All are FALSE by default. DESIGN IN EFFECT shows each
# one as "not used" or with the value in force; a rule in force also tags the results folder (the tag is named beside each option).
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION B -- THE CONTROL GROUP CHOSEN ON THE PRE PERIOD (your request of 30 Sep): which buffers / clusters are the controls for THIS
# outcome, decided on the pre period only and then FIXED across the whole panel (the same control pixels in every year and season)
# --------------------------------------------------------------------------------------------------------------------------------
USE_CONTROL_SELECTION <- FALSE    # FALSE = every ring of CONTROL_RINGS is the control group (as above) | TRUE = CONTROL_SELECTION below chooses it
CONTROL_SELECTION <- "pre_rings"  #   "pre_rings": the CONTROL_SELECT_K (1 or 2) buffers whose PRE-period series is closest to the treatment area's -- your
                                  #   "most close 1 or 2 buffers" | "pre_blocks": control CLUSTERS (~1 km blocks of pixels) from ANY part of the buffers, the
                                  #   closest first, until CONTROL_SELECT_RATIO x the treated pixels -- your "clusters from any part of the buffers".
                                  #   Per outcome; the decision uses the PRE period only (a post- or outcome-based rule is refused); made once and applied
                                  #   to every year and season. Evidence: CONTROL_SELECTION_<outcome>_R.csv beside the results. Tag _ctrlPre2r / _ctrlPreBlk3x
CONTROL_SELECT_K  <- 2L           #   pre_rings: how many buffers -- 1 or 2 (up to 5)
CONTROL_SELECT_RATIO <- 3         #   pre_blocks: control pixels >= this x the treated pixels
CONTROL_SELECT_ON <- "trend"      #   what "closest" means: "trend" = the demeaned pre series' distance (what the parallel-trends assumption asks; recommended)
                                  #   | "level" = the nearest pre-period MEAN (your "mean of the treatment area vs the buffer's mean") | "rmse" = the RMSE of the
                                  #   cell-wise pre differences | "both" = trend + level.  Tag suffix: none / L / R / B
USE_SAME_PIXELS   <- FALSE        # FALSE = a pixel may contribute to one side only (the v20.58 sample) | TRUE = the treated and control groups are the SAME
SAME_PIXELS       <- "pre_post"   #   pixels across the panel: "pre_post" = every pixel observed in pre AND post, else it leaves (tag _pixPP) | "all" = in
                                  #   every year-season of the sample (a balanced pixel set; tag _pixAll). The sample-integrity line confirms it at every run
USE_BALANCED_PANEL <- FALSE      # 4 Oct (your request): FALSE = every pixel is kept where values are missing (an unbalanced panel, as before) |
BALANCED_PANEL    <- "drop"       #   TRUE = applied: "drop" = per variable, a pixel missing the value in ANY year-season of the sample leaves this
                                  #   model's sample (a balanced panel; tag _balanced) | "keep" = kept and counted. The panel file keeps every row
# --------------------------------------------------------------------------------------------------------------------------------
# SECTION C -- PANEL-PREPARATION ENHANCEMENTS (the three specifications of 30 Sep); each with its own switch
# --------------------------------------------------------------------------------------------------------------------------------
USE_DONUT         <- FALSE        # FALSE = ring 1 is a control like the others | TRUE = DONUT_RINGS leave the control pool -- the spillover buffer next to the core
DONUT_RINGS       <- 1L           #   the ring(s) left out, e.g. 1L or c(1L, 2L); applied before any control choice. Tag _donut1
USE_LANDUSE_MASK  <- FALSE        # FALSE = every land-use class | TRUE = only pixels whose PRE-period (baseline) LandUse class is in LANDUSE_KEEP stay. Tag _lu2
LANDUSE_KEEP      <- 2L           #   your exporter's class code(s) for agriculture (a pixel's class is read on the pre period, so the works cannot move it)
USE_BASELINE_NDVI_MASK <- FALSE   # FALSE = no mask | TRUE = only pixels whose PRE-period mean NDVI exceeds BASELINE_NDVI_MIN stay (an agricultural mask). Tag _ndviPre0.25
BASELINE_NDVI_MIN <- 0.25         #   the threshold (an NDVI value)
USE_COVERAGE_THRESHOLD <- FALSE   # FALSE = the standard screen (a year-season below 5 % of the typical coverage leaves) | TRUE = below MIN_PIXEL_COVERAGE_PCT. Tag _cov70
MIN_PIXEL_COVERAGE_PCT <- 0.70    #   the share of the typical treated / control coverage a year-season must reach (your log: 2023 Zaid, 2025 Kharif / Zaid collapsed)
USE_DROP_SINGLETONS <- FALSE      # FALSE = series seen once stay | TRUE = they leave before the demeaning (the pre-flight). Tag _noSingle
USE_PRECISION_TOLERANCE <- FALSE  # FALSE = the no-data zero is an EXACT 0 (the v20.58 rule) | TRUE = |value| <= PRECISION_TOLERANCE is the no-data zero and a
PRECISION_TOLERANCE <- 1e-6       #   year-season is "constant across pixels" within it -- indices in [-1, 1] are never compared with exact equality
# The Rabi-only run of the specifications is SEASONS <- "Rabi" above; the ~1 km block clusters are CLUSTER <- "block" above.
# ================================================================================================================================


**Run** -- load the screened panel, estimate, save (results under ROOT/output/results).

In [ ]:
design <- model_design()      # v20.57: the settings above -> the design of THIS run (DESIGN IN EFFECT)
res <- run_model_R("M45", OUTCOME, design)
if (ALL_OUTCOMES) for (o in setdiff(OUTCOMES, OUTCOME)) run_model_R("M45", o, design)